##In this notebook, we simulate GR4J, GRHyMoLAP and HBV rainfall-runoff models  accross the 8 lowest-yielding CAMELS-AUS catchments.

In [1]:
from google.colab import files

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install git+https://github.com/kratzert/RRMPG.git

  Cloning https://github.com/kratzert/RRMPG.git to /tmp/pip-req-build-m0puudco
  Running command git clone --filter=blob:none --quiet https://github.com/kratzert/RRMPG.git /tmp/pip-req-build-m0puudco
  Resolved https://github.com/kratzert/RRMPG.git to commit 7de78c25acc1c255d2acaf739d65e9ce7bbd60c3
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 3.9 MB/s eta 0:00:00
  Created wheel for rrmpg: filename=rrmpg-0.1.1-py3-none-any.whl size=609986 sha256=67eb3de49b36af50513ef36b942b1e5ffce3409b6fe8f1b38853876a7bd19a4b
  Stored in directory: /tmp/pip-ephem-wheel-cache-_ty3zqoe/wheels/7d/ec/25/408ea8f9d8a1aff931ffd2c082074611b43cc13f5ec46e46b3
Successfully built rrmpg


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import random
from datetime import datetime, timedelta
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from rrmpg.models import GR4J
import os
import zipfile
from scipy.optimize import minimize

from google.colab import files
from numba import njit

#Data from my google drive

In [5]:
# ==============================
# ZIP file paths
# ==============================
hydro_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/05_hydrometeorology.zip"
streamflow_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/03_streamflow.zip"

# Temporary extraction directories
hydro_dir = "/content/05_hydro"
streamflow_dir = "/content/03_streamflow"

# ==============================
# Function to extract ZIP files and locate the CSV path
# ==============================
def extract_zip(zip_path, extract_to):
    if not os.path.exists(extract_to):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✅ ZIP extracted to {extract_to}")
    else:
        print(f"✅ Directory {extract_to} already exists")

def find_csv(base_dir, csv_name):
    # Recursive search for the CSV file
    for root, dirs, files in os.walk(base_dir):
        if csv_name in files:
            return os.path.join(root, csv_name)
    raise FileNotFoundError(f"{csv_name} not found in {base_dir}")

# ==============================
# Extract ZIP files
# ==============================
extract_zip(hydro_zip, hydro_dir)
extract_zip(streamflow_zip, streamflow_dir)

# ==============================
# Load the 222 official stations
# ==============================
file_path = '/content/drive/MyDrive/Colab Notebooks/Dimension/id_name_metadata.csv'
basin222 = pd.read_csv(file_path)
station_ids_v1 = basin222['station_id'].astype(str).str.strip().unique()
print(f"✅ {len(station_ids_v1)} official basins loaded")

# ==============================
# 1️⃣ Precipitation (SILO)
# ==============================
precip_file = find_csv(hydro_dir, "precipitation_SILO.csv")
precip = pd.read_csv(precip_file, index_col=0, parse_dates=True)
precip.columns = precip.columns.str.strip()
precip.replace(-99.99, np.nan, inplace=True)
print("✅ SILO precipitation:", precip.shape)

# ==============================
# 2️⃣ Evapotranspiration (SILO ET)
# ==============================
et_file = find_csv(hydro_dir, "et_morton_actual_SILO.csv")
et = pd.read_csv(et_file, index_col=0, parse_dates=True)
et.columns = et.columns.str.strip()
et.replace(-99.99, np.nan, inplace=True)
print("✅ SILO ET:", et.shape)

# ==============================
# 3️⃣ Streamflow
# ==============================
streamflow_file = find_csv(streamflow_dir, "streamflow_mmd.csv")
Q = pd.read_csv(streamflow_file, index_col=0, parse_dates=True)
Q.columns = Q.columns.str.strip()
Q.replace(-99.99, np.nan, inplace=True)
print("✅ Streamflow:", Q.shape)

# ==============================
# 4️⃣ Identify common stations
# ==============================
stations_precip = set(precip.columns)
stations_et = set(et.columns)
stations_Q = set(Q.columns)

common_stations = [
    s for s in station_ids_v1
    if s in stations_precip and s in stations_et and s in stations_Q
]
print(f"✅ Official common stations: {len(common_stations)}")

# ==============================
# 5️⃣ Subset to common stations
# ==============================
precip = precip[common_stations]
et = et[common_stations]
Q = Q[common_stations]

# ==============================
# 6️⃣ Final check
# ==============================
print("Precipitation:", precip.shape)
print("ET:", et.shape)
print("Streamflow:", Q.shape)
print("Stations (first 10):", common_stations[:10], "...")


✅ ZIP extracted to /content/05_hydro
✅ ZIP extracted to /content/03_streamflow
✅ 222 official basins loaded
✅ SILO precipitation: (43464, 224)
✅ SILO ET: (43464, 224)
✅ Streamflow: (23376, 224)
✅ Official common stations: 222
Precipitation: (43464, 222)
ET: (43464, 222)
Streamflow: (23376, 222)
Stations (first 10): ['912101A', '912105A', '915011A', '917107A', '919003A', '919201A', '919309A', '922101B', '925001A', '926002A'] ...


# RR computation

In [6]:
# =============================================================
# 📌 COMPUTE RR
# =============================================================
RR = []
stations_list = []

start_date = "1980-01-01"
end_date = "2014-12-31"

for st in common_stations:
    P = precip[st].loc[start_date:end_date].to_numpy(float)
    Qs = Q[st].loc[start_date:end_date].to_numpy(float)

    mask = (~np.isnan(P)) & (~np.isnan(Qs))

    if mask.sum() == 0 or np.nansum(P[mask]) <= 0:
        RR.append(np.nan)
    else:
        RR.append(np.nansum(Qs[mask]) / np.nansum(P[mask]))

    stations_list.append(st)

RR = np.array(RR)
stations = np.array(stations_list)

# =============================================================
# 📌 CLEAN
# =============================================================
mask = np.isfinite(RR)
RR = RR[mask]
stations = stations[mask]

# =============================================================
# 📌 PERCENTILES
# =============================================================
p10 = np.percentile(RR, 10)
p50 = np.percentile(RR, 50)
p60 = np.percentile(RR, 60)
p90 = np.percentile(RR, 90)

# =============================================================
# 📌 SELECT CLASSES
# =============================================================
dry_mask = RR <= p10
mid_mask = (RR >= p50) & (RR <= p60)
wet_mask = RR >= p90

dry_stations_all = stations[dry_mask]
mid_stations_all = stations[mid_mask]
wet_stations_all = stations[wet_mask]

dry_RR_all = RR[dry_mask]
mid_RR_all = RR[mid_mask]
wet_RR_all = RR[wet_mask]

# =============================================================
# 📌 SORT INSIDE EACH CLASS (IMPORTANT)
# =============================================================
def sort_pair(st, rr):
    order = np.argsort(rr)
    return st[order], rr[order]

dry_stations_all, dry_RR_all = sort_pair(dry_stations_all, dry_RR_all)
mid_stations_all, mid_RR_all = sort_pair(mid_stations_all, mid_RR_all)
wet_stations_all, wet_RR_all = sort_pair(wet_stations_all, wet_RR_all)

# =============================================================
# 📌 FIXED SIZE SAMPLING
# =============================================================
dry_n = 23
mid_n = 22
wet_n = 22

assert len(dry_stations_all) >= dry_n, "Not enough dry basins"
assert len(mid_stations_all) >= mid_n, "Not enough mid basins"
assert len(wet_stations_all) >= wet_n, "Not enough wet basins"

dry_stations = dry_stations_all[:dry_n]
dry_RR = dry_RR_all[:dry_n]

mid_stations = mid_stations_all[:mid_n]
mid_RR = mid_RR_all[:mid_n]

wet_stations = wet_stations_all[:wet_n]
wet_RR = wet_RR_all[:wet_n]

# =============================================================
# 📌 FINAL CHECK
# =============================================================
print("🟤 Dry:", len(dry_stations))
print("🟡 Mid:", len(mid_stations))
print("🔵 Wet:", len(wet_stations))

# =============================================================
# 📌 OUTPUT
# =============================================================
RR_classes = {
    "dry_0_10": {"stations": dry_stations, "RR": dry_RR},
    "mid_50_60": {"stations": mid_stations, "RR": mid_RR},
    "wet_90_100": {"stations": wet_stations, "RR": wet_RR},
}

🟤 Dry: 23
🟡 Mid: 22
🔵 Wet: 22


In [7]:
# =============================================================
# 📌 DISPLAY RR RANGES
# =============================================================
def print_range(name, rr_values):
    print(f"{name} RR range: [{np.min(rr_values):.4f} , {np.max(rr_values):.4f}]")

print("\n===== RR RANGES =====")
print_range("🟤 Dry (0–10%)", dry_RR)
print_range("🟡 Mid (50–60%)", mid_RR)
print_range("🔵 Wet (90–100%)", wet_RR)


===== RR RANGES =====
🟤 Dry (0–10%) RR range: [0.0060 , 0.0659]
🟡 Mid (50–60%) RR range: [0.1838 , 0.2306]
🔵 Wet (90–100%) RR range: [0.4444 , 0.7955]


#GR4J

In [8]:
print(type(dry_stations), dry_stations[:10])
print(type(mid_stations), mid_stations[:3])
print(type(wet_stations), wet_stations[-2:])

<class 'numpy.ndarray'> ['616002' '616013' 'A2390523' '408200' 'A0030501' 'A2390531' '614044'
 'A2390519' '616065' '210006']
<class 'numpy.ndarray'> ['G8110016' 'A5040523' '212260']
<class 'numpy.ndarray'> ['308799' '113004A']


In [9]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import copy
from google.colab import files

# ============================================
# Parameters
# ============================================
start_date = "1980-01-01"
end_date = "2014-12-31"
b1_ratio = 0.7
max_missing_ratio = 1.0

stations = ['616002', '616013', 'A2390523', '408200', 'A0030501', 'A2390531', '614044',
            'A2390519']

results_GR4J = {}

# ============================================
# Metrics
# ============================================
def NSE(obs, sim):
    df = pd.DataFrame({"obs": obs, "sim": sim}).dropna()
    if df.empty or df["obs"].var() == 0:
        return np.nan
    return 1 - np.sum((df["sim"] - df["obs"])**2) / np.sum((df["obs"] - df["obs"].mean())**2)

def NNSE(obs, sim):
    nse = NSE(obs, sim)
    return 1 / (2 - nse) if not np.isnan(nse) else np.nan

def runoff_ratio(Q, P):
    df = pd.DataFrame({"Q": Q, "P": P}).dropna()
    if df.empty or df["P"].sum() == 0:
        return np.nan
    return df["Q"].sum() / df["P"].sum()

def objective_gr4j(x, Q, P, ET):
    model = GR4J()
    params = dict(zip(model.get_parameter_names(), x))
    model.set_params(params)
    try:
        Qsim = model.simulate(P, ET).flatten()
        nse = NSE(Q, Qsim)
        return 1 - nse if np.isfinite(nse) else 1e6
    except Exception:
        return 1e6

bounds = [
    (1, 3200),    # x1
    (-15, 15),    # x2
    (1, 1000),    # x3
    (0.5, 5)      # x4
]

# ============================================
# Storage for all basins
# ============================================
all_cal = []
all_val = []

# ============================================
# Main loop over basins
# ============================================
for i, station_id in enumerate(stations, start=1):
    Q_obs = Q[station_id].loc[start_date:end_date].to_numpy(float)
    P = precip[station_id].loc[start_date:end_date].to_numpy(float)
    ET = et[station_id].loc[start_date:end_date].to_numpy(float)

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        print("⚠️ Station skipped (no valid data)")
        continue

    missing_ratio = np.isnan(Q_obs).sum() / N
    if missing_ratio > max_missing_ratio:
        print(f"⚠️ Too much missing data ({missing_ratio*100:.1f}%)")
        continue

    RR = runoff_ratio(Q_obs, P)

    # Split calibration/validation
    b1 = int(N * b1_ratio)
    Q_cal, Q_val = Q_obs[:b1], Q_obs[b1:]
    P_cal, P_val = P[:b1], P[b1:]
    ET_cal, ET_val = ET[:b1], ET[b1:]

    # Multi-start calibration
    best_fun = np.inf
    best_x = None
    np.random.seed(42)
    for _ in range(100):
        x0 = [np.random.uniform(b[0], b[1]) for b in bounds]
        res = minimize(objective_gr4j, x0, args=(Q_cal, P_cal, ET_cal),
                       method="L-BFGS-B", bounds=bounds, options={"maxiter":3000})
        if res.fun < best_fun:
            best_fun = res.fun
            best_x = res.x

    if best_x is None:
        print("⚠️ Calibration failed")
        continue

    model = GR4J()
    params = dict(zip(model.get_parameter_names(), best_x))
    model.set_params(params)

    Qsim_cal = model.simulate(P_cal, ET_cal).flatten()
    Qsim_val = model.simulate(P_val, ET_val).flatten()

    nse_cal = NSE(Q_cal, Qsim_cal)
    nse_val = NSE(Q_val, Qsim_val)

    results_GR4J[station_id] = {
        "RR": RR,
        "NSE_cal": nse_cal,
        "NSE_val": nse_val,
        "params": params,
        "Qsim_cal": Qsim_cal,
        "Qsim_val": Qsim_val
    }

    # Storage for pivot
    dates_cal = pd.date_range(start=start_date, periods=b1, freq="D")
    dates_val = pd.date_range(start=start_date, periods=N, freq="D")[b1:]

    all_cal.append(pd.DataFrame({"Date": dates_cal, "Basin": station_id, "Pred": Qsim_cal}))
    all_val.append(pd.DataFrame({"Date": dates_val, "Basin": station_id, "Pred": Qsim_val}))

    print(f"[{i}/{len(stations)}] Basin {station_id} | NSE_cal: {nse_cal:.3f} | NSE_val: {nse_val:.3f} | RR: {RR:.3f}")

# ============================================
# Pivot for Excel
# ============================================
df_cal = pd.concat(all_cal).pivot(index="Date", columns="Basin", values="Pred").reset_index()
df_val = pd.concat(all_val).pivot(index="Date", columns="Basin", values="Pred").reset_index()

# ============================================
# Export to Excel + download
# ============================================
file_name = "GR4J_cal_val_all_basins.xlsx"
with pd.ExcelWriter(file_name) as writer:
    df_cal.to_excel(writer, sheet_name="Calibration", index=False)
    df_val.to_excel(writer, sheet_name="Validation", index=False)

print(f"\n📁 Excel saved: {file_name}")
#files.download(file_name)

[1/8] Basin 616002 | NSE_cal: 0.444 | NSE_val: 0.385 | RR: 0.006
[2/8] Basin 616013 | NSE_cal: 0.722 | NSE_val: 0.433 | RR: 0.008
[3/8] Basin A2390523 | NSE_cal: 0.395 | NSE_val: -0.781 | RR: 0.012
[4/8] Basin 408200 | NSE_cal: 0.799 | NSE_val: -1.058 | RR: 0.013
[5/8] Basin A0030501 | NSE_cal: 0.213 | NSE_val: 0.144 | RR: 0.014
[6/8] Basin A2390531 | NSE_cal: 0.658 | NSE_val: 0.309 | RR: 0.016
[7/8] Basin 614044 | NSE_cal: 0.421 | NSE_val: -0.001 | RR: 0.019
[8/8] Basin A2390519 | NSE_cal: 0.833 | NSE_val: 0.354 | RR: 0.023

📁 Excel saved: GR4J_cal_val_all_basins.xlsx


##GRHyMoLAP

In [10]:
# ============================================
# General parameters
# ============================================
start_date = "1980-01-01"
end_date = "2014-12-31"
b1_ratio = 0.7
max_missing_ratio = 1.0

stations = ['616002', '616013', 'A2390523', '408200', 'A0030501', 'A2390531', '614044',
            'A2390519']
results_GRHyMoLAP = {}

# Directory to save simulated series
output_dir = "/content/GRHyMoLAP_Qsim"
os.makedirs(output_dir, exist_ok=True)

# ============================================
# NSE definition
# ============================================
@njit
def NSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0 or np.var(obs) == 0:
        return np.nan
    return 1 - np.sum((sim - obs)**2) / np.sum((obs - np.mean(obs))**2)

# ============================================
# Percolation function
# ============================================
@njit
def Percolation(Pn, En, X1):
    n = len(Pn)
    S = np.zeros(n)
    S[0] = X1 / 2
    Perc = np.zeros(n)

    ratio = (4.0 / 9.0) * (S[0] / X1)
    Perc[0] = S[0] * (1 - (1 + ratio**4) ** (-0.25))

    for i in range(1, n):
        temp = (S[i-1] / X1) ** 2
        frac = Pn[i] / X1
        Ps = X1 * (1 - temp) * np.tanh(frac) / (1 + (S[i-1] / X1) * np.tanh(frac))

        frac = En[i] / X1
        Es = S[i-1] * (2 - S[i-1]/X1) * np.tanh(frac) / (1 + (1 - S[i-1]/X1) * np.tanh(frac))

        S[i] = S[i-1] + Ps - Es

        ratio = (4.0 / 9.0) * (S[i] / X1)
        Perc[i] = S[i] * (1 - (1 + ratio**4) ** (-0.25))
        S[i] -= Perc[i]

    return Perc

# ============================================
# GRHyMoLAP simulation
# ============================================
@njit
def GRHyMoLAP_Model(params, Q0, Pn, En):
    MU, LAMBDA, X1, GAMMA = params
    N = len(Pn)

    Q = np.zeros(N)
    Q[0] = Q0

    Perc = Percolation(Pn, En, X1)

    for t in range(N-1):
        Q[t+1] = max(
            0,
            Q[t]
            - (MU / LAMBDA) * Q[t]**(2*MU - 1)
            + GAMMA * Perc[t+1] * Pn[t+1]
        )

    return Q

# ============================================
# Main loop
# ============================================
all_cal = []
all_val = []

for i, station_id in enumerate(stations, start=1):

    Q_obs = Q[station_id].loc[start_date:end_date].to_numpy(float)
    P = precip[station_id].loc[start_date:end_date].to_numpy(float)
    PET = et[station_id].loc[start_date:end_date].to_numpy(float)

    Pn = np.maximum(0, P - PET)
    En = np.maximum(0, PET - P)

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        continue

    missing_ratio = np.sum(np.isnan(Q_obs)) / N
    if missing_ratio > max_missing_ratio:
        continue

    # Split
    b1 = int(N * b1_ratio)
    Q0 = Q_obs[0]

    # Objective
    def objective(params, Q0, Pn_train, En_train, Q_obs_train):
        Q_sim = GRHyMoLAP_Model(params, Q0, Pn_train, En_train)
        nse = NSE(Q_obs_train, Q_sim)
        return 1 - nse if np.isfinite(nse) else 1e9

    # Multi-start Nelder-Mead
    initial_guesses = [
        [1.0, 8, 150, 0.1],
        [0.6, 2, 120, 1.0],
        [1.4, 15, 200, 0.5],
        [0.8, 4, 500, 0.2],
        [1.2, 10, 1000, 0.3],
        [0.7, 6, 350, 0.4],
        [1.1, 18, 20, 0.2],
        [0.9, 8, 2000, 0.6],
    ]

    best_val = float("inf")
    best_res = None

    for guess in initial_guesses:
        res = minimize(
            objective,
            guess,
            args=(Q0, Pn[:b1], En[:b1], Q_obs[:b1]),
            method="Nelder-Mead",
            options={'maxiter': 2500, 'disp': False}
        )
        if res.fun < best_val:
            best_val = res.fun
            best_res = res

    if best_res is None:
        continue

    MU, LAMBDA, X1, GAMMA = best_res.x

    # Full simulation
    Qsim = GRHyMoLAP_Model([MU, LAMBDA, X1, GAMMA], Q0, Pn, En)

    # NSE
    NSE_cal = 1 - best_res.fun
    NSE_val = NSE(Q_obs[b1:], Qsim[b1:])

    # PRINT
    print(
        f"[{i}/{len(stations)}] Station {station_id} | "
        f"NSE_cal: {NSE_cal:.3f} | NSE_val: {NSE_val:.3f}"
    )

    # Storage for Excel
    all_cal.append(pd.DataFrame({
        "Date": pd.date_range(start=start_date, periods=b1, freq="D"),
        "Basin": station_id,
        "Pred": Qsim[:b1]
    }))
    all_val.append(pd.DataFrame({
        "Date": pd.date_range(start=start_date, periods=N, freq="D")[b1:],
        "Basin": station_id,
        "Pred": Qsim[b1:]
    }))

    # Minimal storage
    results_GRHyMoLAP[station_id] = {
        "NSE_cal": NSE_cal,
        "NSE_val": NSE_val,
        "params": [MU, LAMBDA, X1, GAMMA]
    }

# ==========================================================
# Pivot and export to Excel
# ==========================================================
df_cal = pd.concat(all_cal).pivot(index="Date", columns="Basin", values="Pred").reset_index()
df_val = pd.concat(all_val).pivot(index="Date", columns="Basin", values="Pred").reset_index()

file_name = "GRHyMoLAP_Qsim.xlsx"
with pd.ExcelWriter(file_name) as writer:
    df_cal.to_excel(writer, sheet_name="Calibration", index=False)
    df_val.to_excel(writer, sheet_name="Validation", index=False)

print(f"\n📁 Excel file saved: {file_name}")
#files.download(file_name)

print(f"\n✅ Simulation completed for {len(results_GRHyMoLAP)} basins.")

[1/8] Station 616002 | NSE_cal: -0.046 | NSE_val: -0.048
[2/8] Station 616013 | NSE_cal: 0.702 | NSE_val: 0.567
[3/8] Station A2390523 | NSE_cal: 0.811 | NSE_val: 0.463
[4/8] Station 408200 | NSE_cal: 0.534 | NSE_val: 0.131
[5/8] Station A0030501 | NSE_cal: 0.344 | NSE_val: 0.401
[6/8] Station A2390531 | NSE_cal: 0.691 | NSE_val: 0.282
[7/8] Station 614044 | NSE_cal: 0.760 | NSE_val: 0.317
[8/8] Station A2390519 | NSE_cal: 0.704 | NSE_val: 0.483

📁 Excel file saved: GRHyMoLAP_Qsim.xlsx

✅ Simulation completed for 8 basins.


##HBV

In [11]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from numba import njit
from google.colab import files

# ============================================
# 1️⃣ General settings
# ============================================
start_date = "1980-01-01"
end_date = "2014-12-31"
b1_ratio = 0.7
max_missing_ratio = 1.0
stations = ['616002', '616013', 'A2390523', '408200', 'A0030501', 'A2390531', '614044', 'A2390519']
results_HBV = {}

# ============================================
# 2️⃣ Metrics with Numba
# ============================================
@njit
def NSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0 or np.var(obs) == 0:
        return np.nan
    return 1 - np.sum((sim - obs)**2) / np.sum((obs - np.mean(obs))**2)

# ============================================
# 3️⃣ HBV model
# ============================================
@njit
def hbv_numba(P, ET, params):
    FC, BETA, LP, K0, K1, K2, UZL, PERC = params
    SM = 0.5 * FC
    UZ = 0.0
    LZ = 0.0
    n = len(P)
    Qsim = np.zeros(n)

    for t in range(n):
        p = P[t]
        et = ET[t]

        soil_ratio = SM / FC
        recharge = p * soil_ratio**BETA
        SM += p - recharge
        if SM > FC:
            recharge += SM - FC
            SM = FC
        evap = min(et * min(SM / (LP * FC), 1.0), SM)
        SM -= evap

        UZ += recharge
        perc = min(PERC, UZ)
        UZ -= perc
        LZ += perc

        Q0 = K0 * (UZ - UZL) if UZ > UZL else 0.0
        if Q0 > 0:
            UZ -= Q0

        Q1 = K1 * UZ
        UZ -= Q1

        Q2 = K2 * LZ
        LZ -= Q2

        Qsim[t] = max(0.0, Q0 + Q1 + Q2)

    return Qsim

# ============================================
# 4️⃣ Objective function
# ============================================
def objective_hbv(x, Q, P, ET):
    Qsim = hbv_numba(P, ET, x)
    nse = NSE(Q, Qsim)
    return 1 - nse if np.isfinite(nse) else 1e6

# ============================================
# 5️⃣ Calibration loop
# ============================================
bounds = [
    (50,1000), (1,6), (0.3,1), (0,0.5), (0,0.3),
    (0,0.1), (0,100), (0,5)
]

# Storage for Excel
all_cal, all_val = [], []

for i, station_id in enumerate(stations, start=1):
    print(f"\n=== Station {station_id} ({i}/{len(stations)}) ===")

    Q_obs = Q[station_id].loc[start_date:end_date].to_numpy(float)
    P = precip[station_id].loc[start_date:end_date].to_numpy(float)
    ET = et[station_id].loc[start_date:end_date].to_numpy(float)

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        continue
    missing_ratio = np.sum(np.isnan(Q_obs)) / N
    if missing_ratio > max_missing_ratio:
        continue

    b1 = int(N * b1_ratio)
    Q_cal, Q_val = Q_obs[:b1], Q_obs[b1:]
    P_cal, P_val = P[:b1], P[b1:]
    ET_cal, ET_val = ET[:b1], ET[b1:]

    # Multi-start calibration
    best_fun = np.inf
    best_x = None
    np.random.seed(42)
    for _ in range(10):
        x0 = [np.random.uniform(b[0], b[1]) for b in bounds]
        res = minimize(
            objective_hbv,
            x0,
            args=(Q_cal, P_cal, ET_cal),
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter":3000}
        )
        if res.fun < best_fun:
            best_fun = res.fun
            best_x = res.x

    if best_x is None:
        continue

    # Simulations
    Qsim_cal = hbv_numba(P_cal, ET_cal, best_x)
    Qsim_val = hbv_numba(P_val, ET_val, best_x)

    # Storage for Excel
    dates_cal = pd.date_range(start=start_date, periods=b1, freq="D")
    dates_val = pd.date_range(start=start_date, periods=N, freq="D")[b1:]

    all_cal.append(pd.DataFrame({"Date": dates_cal, station_id: Qsim_cal}))
    all_val.append(pd.DataFrame({"Date": dates_val, station_id: Qsim_val}))

    print(f"NSE_cal: {NSE(Q_cal, Qsim_cal):.3f}, NSE_val: {NSE(Q_val, Qsim_val):.3f}")

# ============================================
# 6️⃣ Excel export (discharge only)
# ============================================
df_cal = all_cal[0]
df_val = all_val[0]
for df in all_cal[1:]:
    df_cal = df_cal.merge(df, on="Date", how="outer")
for df in all_val[1:]:
    df_val = df_val.merge(df, on="Date", how="outer")

file_name = "HBV_Qsim_only.xlsx"
with pd.ExcelWriter(file_name) as writer:
    df_cal.to_excel(writer, sheet_name="Calibration", index=False)
    df_val.to_excel(writer, sheet_name="Validation", index=False)

print(f"\n📁 Excel file saved: {file_name}")
#files.download(file_name)


=== Station 616002 (1/8) ===
NSE_cal: 0.593, NSE_val: 0.342

=== Station 616013 (2/8) ===
NSE_cal: 0.641, NSE_val: 0.111

=== Station A2390523 (3/8) ===
NSE_cal: 0.679, NSE_val: 0.446

=== Station 408200 (4/8) ===
NSE_cal: 0.529, NSE_val: -0.045

=== Station A0030501 (5/8) ===
NSE_cal: 0.357, NSE_val: 0.268

=== Station A2390531 (6/8) ===
NSE_cal: 0.672, NSE_val: 0.144

=== Station 614044 (7/8) ===
NSE_cal: 0.657, NSE_val: 0.335

=== Station A2390519 (8/8) ===
NSE_cal: 0.679, NSE_val: 0.342

📁 Excel file saved: HBV_Qsim_only.xlsx
